# MLOps Lab 2 — Hyperparameter Tuning with MLflow

**Dataset:** `kidney_disease_cleaned.csv`

### Learning Objectives
- Understand hyperparameters.
- Establish a baseline model.
- Perform grid-search style tuning.
- Track each configuration with MLflow.
- Compare baseline and tuned performance.
- Visualize tuning results.

## 1. Theory

A **hyperparameter** is a setting chosen before training.

Examples:

```text
Random Forest → n_estimators, max_depth
Logistic Regression → C
SVM → C, kernel
```

Workflow:

```text
Baseline
   ↓
Define Search Space
   ↓
Train Configurations
   ↓
Evaluate
   ↓
Track Runs
   ↓
Compare
   ↓
Best Configuration
```

This lab uses `ParameterGrid`, which generates every combination in a defined grid.

## 2. Setup

Install:

```bash
python -m pip install mlflow scikit-learn pandas matplotlib seaborn
```

Start the UI separately:

```bash
mlflow ui
```

Open `http://127.0.0.1:5000`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

print("Libraries imported successfully.")

In [ ]:
df = pd.read_csv("kidney_disease_cleaned.csv")

X = df.drop("classification", axis=1).copy()
y = df["classification"].astype(str).str.strip()

if "id" in X.columns:
    X = X.drop("id", axis=1)

num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

mlflow.set_experiment("Kidney Disease - Hyperparameter Tuning")

## 3. Baseline Model

Always establish a baseline before tuning.

Here the baseline is Random Forest with:

```text
n_estimators = 100
max_depth = None
```

In [ ]:
baseline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100, random_state=42
    ))
])

with mlflow.start_run(run_name="RF Baseline"):
    mlflow.log_params({
        "model": "Random Forest",
        "n_estimators": 100,
        "max_depth": "None",
        "random_state": 42
    })

    baseline.fit(X_train, y_train)
    pred = baseline.predict(X_test)

    baseline_metrics = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, average="weighted"),
        "recall": recall_score(y_test, pred, average="weighted"),
        "f1_score": f1_score(y_test, pred, average="weighted")
    }

    mlflow.log_metrics(baseline_metrics)

print(baseline_metrics)

## 4. Define the Search Grid

Example:

```text
n_estimators = [50, 100, 150]
max_depth    = [5, 10, 15]

3 × 3 = 9 configurations
```

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [5, 10, 15]
}

grid = list(ParameterGrid(param_grid))

print("Configurations:", len(grid))
display(pd.DataFrame(grid))

## 5. Run the Grid Search and Track Every Configuration

Each configuration becomes an MLflow run.

This is the key MLOps connection:

```text
Hyperparameter Configuration
          ↓
       MLflow Run
          ↓
Parameters + Metrics
```

In [ ]:
results = []

for i, params in enumerate(grid, start=1):

    model = Pipeline([
        ("preprocessing", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            random_state=42
        ))
    ])

    with mlflow.start_run(run_name=f"RF Grid Run {i}"):

        mlflow.log_params({
            "model": "Random Forest",
            **params,
            "random_state": 42
        })

        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        metrics = {
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, average="weighted"),
            "recall": recall_score(y_test, pred, average="weighted"),
            "f1_score": f1_score(y_test, pred, average="weighted")
        }

        mlflow.log_metrics(metrics)

        results.append({**params, **metrics})

tuning_results = pd.DataFrame(results)
display(tuning_results.sort_values("f1_score", ascending=False))

## 6. Select the Best Configuration

For this classroom example, F1-score is used as the selection metric.

In real projects, the selection criterion depends on the application.

In [ ]:
best = tuning_results.loc[tuning_results["f1_score"].idxmax()]

print("Best configuration:")
display(best.to_frame("value"))

## 7. Compare Baseline vs Tuned Model

In [ ]:
summary = pd.DataFrame({
    "Model": ["Baseline", "Best Tuned"],
    "Accuracy": [
        baseline_metrics["accuracy"], best["accuracy"]
    ],
    "F1 Score": [
        baseline_metrics["f1_score"], best["f1_score"]
    ]
})

display(summary)

print(
    "F1 improvement:",
    best["f1_score"] - baseline_metrics["f1_score"]
)

## 8. Visualize the Search

A heatmap makes it easy to see which hyperparameter combination produced stronger results.

In [ ]:
heatmap_data = tuning_results.pivot(
    index="max_depth",
    columns="n_estimators",
    values="f1_score"
)

plt.figure(figsize=(8,6))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="Blues")
plt.title("Random Forest Hyperparameter Search — F1 Score")
plt.xlabel("n_estimators")
plt.ylabel("max_depth")
plt.tight_layout()
plt.show()

In [ ]:
# Save the full search table and log it as an artifact.
tuning_results.to_csv("hyperparameter_tuning_results.csv", index=False)

with mlflow.start_run(run_name="Tuning Summary"):
    mlflow.log_param("search_method", "Grid Search")
    mlflow.log_param("configurations_tested", len(grid))
    mlflow.log_param("selection_metric", "f1_score")
    mlflow.log_metric("baseline_f1", float(baseline_metrics["f1_score"]))
    mlflow.log_metric("best_tuned_f1", float(best["f1_score"]))
    mlflow.log_artifact("hyperparameter_tuning_results.csv")

## 9. Student Practice

### Task 1
Expand the grid:

```python
n_estimators = [50, 100, 150, 200]
max_depth = [3, 5, 10, 15, 20]
```

How many runs are produced?

### Task 2
Tune SVM using:

```text
C = [0.1, 1, 10]
kernel = ["linear", "rbf"]
```

### Task 3
Compare the best configuration using Accuracy, Precision, Recall and F1.

### Task 4
Use the MLflow UI to identify the best run.

### Questions
1. What is a hyperparameter?
2. Why do we need a baseline?
3. What is the difference between grid search and random search?
4. Why should tuning runs be tracked?
5. Did tuning improve the baseline?

# End of Lab

```text
Baseline
   ↓
Search Space
   ↓
Many Configurations
   ↓
MLflow Runs
   ↓
Compare
   ↓
Best Configuration
```

### Next topic

➡️ **Model Management & Model Registry**